Recovered from the kernel's run history (session of 2026-09-30 16:29): the cells exactly as executed, in execution order.

In [50]:
import regex as re

In [51]:
def get_stats(ids)-> dict: # this is just 
    counts = {}
    for i in range(len(ids)):
        for pair in zip(ids[i], ids[i][1:]):
            counts[pair] = counts.get(pair, 0) + 1
    return counts

In [52]:
def merge(ids, pair, idx):

    out = [[] for _ in range (len(ids))]
    for n in range(len(ids)):
        ids_n = ids[n]
        i = 0
        while i < len(ids_n):
            if i < len(ids_n) - 1 and ids_n[i] == pair[0] and ids_n[i + 1] == pair[1]:
                out[n].append(idx)
                i += 2
            else:
                out[n].append(ids_n[i])
                i += 1
    return out

In [53]:
pattern = re.compile("|".join([
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]+[\p{Ll}\p{Lm}\p{Lo}\p{M}]*(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""\p{N}{1,3}""",
    r""" ?[^\s\p{L}\p{N}]+[\r\n/]*""",
    r"""\s*[\r\n]+""",
    r"""\s+(?!\S)""",
    r"""\s+""",
]))

In [54]:
# ai genertaed 

def get_earliest_merge(token_list, merges):
    """
    Finds the adjacent pair in token_list that has the earliest merge rank
    (lowest integer key) in the merges dictionary.
    
    merges format: {rank_int: [item0, item1]}
    """
    # 1. Invert the dict so we can look up ranks in O(1): (item0, item1) -> rank
    pair_to_rank = {tuple(pair): rank for rank, pair in merges.items()}

    # 2. Extract unique adjacent pairs present in the list
    if token_list and isinstance(token_list[0], (list, tuple)):
        # If the list is already given as candidate pairs
        pairs_in_list = {tuple(p) for p in token_list}
    else:
        # If the list is a sequence of tokens/bytes
        pairs_in_list = {
            (token_list[i], token_list[i + 1])
            for i in range(len(token_list) - 1)
        }

    # 3. Filter for pairs that exist in merges
    valid_candidates = [p for p in pairs_in_list if p in pair_to_rank]

    if not valid_candidates:
        return None, None

    # 4. Pick the pair with the lowest rank (merged first)
    best_pair = min(valid_candidates, key=lambda p: pair_to_rank[p])
    best_rank = pair_to_rank[best_pair]

    return best_rank, list(best_pair)

In [55]:
def train(text, vocab_size, special_tokens,verbose=False): 

    pieces =re.split(re.escape(special_tokens), text) 
    print(f"pieces : {pieces}")
    pre_tokens = [] # This just have no more of the special token 
    for piece in pieces:
        pre_tokens.extend(pattern.findall(piece)) # this create a list of new word

    print(f"pretokens: {pre_tokens}")

    # text = "".join(text)
    #merges: list[tuple[bytes, bytes]]
    ids = list(chunk.encode("utf-8") for chunk in pre_tokens)

    vocab = {i: bytes([i]) for i in range(256)}
 
    print(f"ids {ids}") 
    # 257 : b "cd" 
    merges_list = []
    merges = {} # (97,98) -> 257 

    for idx in range(256, vocab_size - 1 ): # 256 -> 398 
        stats = get_stats(ids)
        if not stats:                                # nothing left to merge
            break
        pair = max(stats, key=lambda p: (stats[p], vocab[p[0]], vocab[p[1]]), )        # tie breaking base on the bytes if want to go ofr the lower use - ? 
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        merges_list.append((vocab[pair[0]], vocab[pair[1]]))
        if verbose:
            print(f"merge {idx - 255}/{vocab_size - 256}: {pair} -> {idx} {vocab[idx]} ({stats[pair]}x)")

        # just do voacg size cuz the loop ends at 399, with vocab sie being 400
    vocab[vocab_size-1] = b'<|endoftext|>' # might need to improve this later this is the lazy way out  
        # vocab[vocab_size -2]
    return vocab, merges_list

In [ ]:
def encode(text):

    pieces = re.split(re.escape("<|endoftext|>"), text)

    # another layer to actually use the regex pattenr using extends cuz 
    pre_tokens = [] 
    for piece in pieces:
        pre_tokens.extend(pattern.findall(piece))

    ids =  [token.encode("utf-8") for token in pre_tokens]
    # dont forget that token is a nested loop so we need to loop inside it 
    for idx in range(len(ids)):
        stats = get_stats(pre_tokens)   

        # idx is not a [] it's a number 
        i_ids = ids[idx] # ids is just a list of byes 

        for i_idx in i_ids:
            for i in range(256, len(merges)):
                if (merges[i][0] == i_ids[i_idx] and merges[i+1] == i_ids[i_idx+1]): # might have some out of loop error
                    pair = merges[i]
                    ids = merge(ids, pair, merges[i])
                 
    return ids 


In [ ]:
def encode(text):

    # encode from text to remove the special = [ ]
    special_token = "<|endoftext|>"

    pattern = f"({re.escape(special_token)})"
    chunks = re.split(pattern, text)

    # another layer to actually use the regex pattenr using extends cuz 
    pre_tokens = [] 
    for piece in pieces:
        pre_tokens.extend(pattern.findall(piece))

    ids = []

        

    ids = [token.encode("utf-8") for token in pre_tokens if token == type(bytes)] 


    new_ids = []
    # dont forget that token is a nested loop so we need to loop inside it 
    # if  token_bytes[]== b"endoftext"
    # dont merge just attach an id on to it 
    # but do include it in the ids? 

    for token_index in range(len(ids)): # at chunks 
        
        token_bytes = ids[token_index] # for chunk 
       
        if len(token_bytes) >= 2 : 

            while True : 
        
                chunk, best_rank = get_earliest_merge(token_bytes,merges) 

                if best_rank == None  or len(chunk) ==0:
                   break 
                else:
                    pair = merges[best_rank] 
                    token_bytes = merge(chunk, pair, best_rank)  
            new_ids.extend(token_bytes)
        else :
            new_ids.extend(token_bytes)        
                 
    return new_ids 


In [58]:
def decode(ids):
    pass

In [59]:
t = encode("hello i am a cunt <|endoftext|> dick sucker")
print(t)

KeyError: None

In [ ]:
special_tokens = "<|endoftext|>"

with open("../data/small.txt", "r", encoding="utf-8") as f:
    text = f.read()

vocab, merg = train(text, 400,special_tokens)

print(merges)

pieces : ['u don\'t have to be scared of the loud dog, I\'ll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.\n', '\nOnce upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.\nTom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."\nSam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."\nThey went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but no one cou

In [ ]:
sorted(vocab) == list(range(400))
print(len(vocab))

400


In [ ]:
len(merges)

143

In [ ]:
vocab[399]

b'<|endoftext|>'

In [ ]:
len(vocab) == 400

True

In [ ]:
any(b"<|" in v for k, v in vocab.items() if k != 399)


False

In [ ]:
text = "cd cd abab"

vocab, merges=train(text,262,"<|endoftext|>")
print(len(merges))
print(len(vocab))

pieces : ['cd cd abab']
pretokens: ['cd', ' cd', ' abab']
ids [b'cd', b' cd', b' abab']
5
261


In [ ]:
sorted(vocab) == list(range(262))

False

In [ ]:
any(b"<|" in v for k, v in vocab.items() if k != 262)

True

In [ ]:
print(vocab[260])

b'<|endoftext|>'


In [ ]:
%pwd

'/Users/cheapanharith/AI/standform-LLMSCRATCH/idontwanttobehomelesstoo/cs336_basics'

In [ ]:

test = {1 : [0,1]}
print(test[1][1])

1


In [ ]:
merges[1]


(b'h', b'e')

In [ ]:
vocab[257]

b'he'

In [ ]:
# ai genertaed 

def get_earliest_merge(token_list, merges):
    """
    Finds the adjacent pair in token_list that has the earliest merge rank
    (lowest integer key) in the merges dictionary. 
    merges format: {rank_int: [item0, item1]}
    """
    # 1. Invert the dict so we can look up ranks in O(1): (item0, item1) -> rank
    pair_to_rank = {tuple(pair): rank for rank, pair in merges.items()}

    # 2. Extract unique adjacent pairs present in the list
    if token_list and isinstance(token_list[0], (list, tuple)):
        # If the list is already given as candidate pairs
        pairs_in_list = {tuple(p) for p in token_list}
    else:
        # If the list is a sequence of tokens/bytes
        pairs_in_list = {
            (token_list[i], token_list[i + 1])
            for i in range(len(token_list) - 1)
        }

    # 3. Filter for pairs that exist in merges
    valid_candidates = [p for p in pairs_in_list if p in pair_to_rank]

    if not valid_candidates:
        return None, None

    # 4. Pick the pair with the lowest rank (merged first)
    best_pair = min(valid_candidates, key=lambda p: pair_to_rank[p])
    best_rank = pair_to_rank[best_pair]

    return best_rank, list(best_pair)

Chosen Rank: 1
Chosen Pair: [b't', b'c']


In [ ]:
t = [ 1,3,4,5,6]

print(t)

AttributeError: 'list' object has no attribute 'push'